# 02 · Diagnostics: why?

`01_eda` described what exists. This notebook digs into **why**: where the gaps are and how big,
whether the team's plan targets them, why some content has no usage data, how teacher content
distorts averages, and how clean DIKSHA's language labels are.

In [ ]:
from google.cloud import bigquery
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT = "learning-reach-473210"
client = bigquery.Client(project=PROJECT, location="asia-south1")

def q(sql: str) -> pd.DataFrame:
    """Run SQL in BigQuery and return a DataFrame (INT64 -> float, see 01_eda)."""
    df = client.query(sql).to_dataframe()
    int_cols = df.select_dtypes(include=["Int64"]).columns
    return df.astype({c: "float64" for c in int_cols})

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.3f}".format)
plt.rcParams.update({"figure.figsize": (10, 5), "axes.spines.top": False, "axes.spines.right": False})
FOCUS = ["Hindi", "Marathi", "Gujarati", "Kannada", "Punjabi", "Assamese"]

## 1. Biggest gaps per focus language

Two views (`marts.mart_supply_demand`, within each language):
- **gap_score** = demand share − supply share → big absolute gaps (many plays, relatively few items)
- **demand_supply_ratio** → intense small gaps (e.g. 5 items with 65K plays each); cells with ≥ 10 resources

In [ ]:
gaps = q("""
SELECT language, board, grade_label, subject_group, n_resources,
       sessions_per_resource, demand_share, supply_share, gap_score, demand_supply_ratio
FROM `learning-reach-473210.marts.mart_supply_demand`
WHERE is_focus_language
""")
for lang in FOCUS:
    g = gaps[gaps["language"] == lang]
    top_gap = g.nlargest(5, "gap_score")
    top_ratio = g[g["n_resources"] >= 10].nlargest(5, "demand_supply_ratio")
    print(f"\n=== {lang} ===")
    display(top_gap[["board", "grade_label", "subject_group", "n_resources", "gap_score"]]
            .assign(gap_points=lambda x: 100 * x["gap_score"]).drop(columns="gap_score"))
    display(top_ratio[["board", "grade_label", "subject_group", "n_resources", "demand_supply_ratio"]])

## 2. Is the plan aimed at the gaps? (Localization Plan sheet is SYNTHETIC)

In [ ]:
plan = q("""
SELECT alignment, COUNT(*) AS plans, SUM(resources_planned) AS resources_planned
FROM `learning-reach-473210.marts.mart_plan_vs_gap` GROUP BY alignment ORDER BY plans DESC
""")
plan["share_of_effort"] = plan["resources_planned"] / plan["resources_planned"].sum()
display(plan)

unplanned = q("""
SELECT language, gap_rank, board, grade_label, subject_group, n_resources, gap_score
FROM `learning-reach-473210.marts.mart_unplanned_gaps`
WHERE gap_rank <= 3 ORDER BY language, gap_rank
""")
display(unplanned.assign(gap_points=lambda x: 100 * x["gap_score"]).drop(columns="gap_score"))

## 3. Why do some resources have no usage data?

~18% of student resources carry no play count. If that is concentrated in some formats or boards,
usage-based comparisons are biased (selection bias) — important caveat for `03_statistics`.

In [ ]:
cov = q("""
SELECT format, board_type, language_group, COUNT(*) AS resources,
       AVG(IF(has_usage, 1, 0)) AS pct_with_usage
FROM `learning-reach-473210.marts.mart_content_features`
WHERE NOT is_teacher_content
GROUP BY CUBE(format, board_type, language_group)
""")
for dim in ["format", "board_type", "language_group"]:
    others = [c for c in ["format", "board_type", "language_group"] if c != dim]
    t = cov[cov[dim].notna() & cov[others].isna().all(axis=1)]
    display(t[[dim, "resources", "pct_with_usage"]].sort_values("pct_with_usage"))

## 4. Teacher content: does it distort the averages?

In [ ]:
teach = q("""
SELECT is_teacher_content, COUNT(*) AS resources,
       APPROX_QUANTILES(play_sessions, 2)[OFFSET(1)] AS median_plays,
       AVG(play_sessions) AS mean_plays, SUM(play_sessions) AS total_plays
FROM `learning-reach-473210.marts.mart_content_features`
WHERE has_usage GROUP BY is_teacher_content
""")
teach["share_of_all_plays"] = teach["total_plays"] / teach["total_plays"].sum()
teach

## 5. Label quality: does the title's script match the medium?

A resource tagged `medium = Marathi` should usually have a title in Devanagari (or Latin, which is
common for English titles). A title in **another Indian script** (e.g. Gurmukhi for a "Hindi" item, like
"8TH SCIENCE PUNJABI" seen on day 1) suggests a mislabel. Single-language resources only.

In [ ]:
mismatch = q("""
WITH titled AS (
  SELECT m.language, m.script, c.name,
    CASE
      WHEN REGEXP_CONTAINS(c.name, r'\\p{Devanagari}') THEN 'Devanagari'
      WHEN REGEXP_CONTAINS(c.name, r'\\p{Gurmukhi}') THEN 'Gurmukhi'
      WHEN REGEXP_CONTAINS(c.name, r'\\p{Gujarati}') THEN 'Gujarati'
      WHEN REGEXP_CONTAINS(c.name, r'\\p{Kannada}') THEN 'Kannada'
      WHEN REGEXP_CONTAINS(c.name, r'\\p{Bengali}') THEN 'Bengali-Assamese'
      WHEN REGEXP_CONTAINS(c.name, r'\\p{Tamil}') THEN 'Tamil'
      WHEN REGEXP_CONTAINS(c.name, r'\\p{Telugu}') THEN 'Telugu'
      WHEN REGEXP_CONTAINS(c.name, r'\\p{Oriya}') THEN 'Odia'
      WHEN REGEXP_CONTAINS(c.name, r'\\p{Malayalam}') THEN 'Malayalam'
      WHEN REGEXP_CONTAINS(c.name, r'\\p{Arabic}') THEN 'Perso-Arabic'
      ELSE 'Latin/other'
    END AS title_script
  FROM `learning-reach-473210.staging.stg_diksha_content` AS c
  JOIN `learning-reach-473210.staging.stg_diksha_medium` AS m USING (identifier)
  WHERE m.identifier IN (
    SELECT identifier FROM `learning-reach-473210.staging.stg_diksha_medium`
    GROUP BY identifier HAVING COUNT(*) = 1)
    AND m.script IS NOT NULL AND m.language != 'English'
)
SELECT language, script AS expected_script,
       COUNT(*) AS resources,
       COUNTIF(title_script = script) AS title_in_expected_script,
       COUNTIF(title_script = 'Latin/other') AS title_in_latin,
       COUNTIF(title_script NOT IN (script, 'Latin/other')) AS title_in_other_indian_script
FROM titled GROUP BY language, script
HAVING resources >= 1000
ORDER BY language
""")
mismatch["pct_other_indian_script"] = 100 * mismatch["title_in_other_indian_script"] / mismatch["resources"]
mismatch.sort_values("pct_other_indian_script", ascending=False)

## 6. YouTube: which videos keep getting views?

Daily history started 2 Oct 2026, so views-per-day becomes meaningful after a few days.
Until then, lifetime views per day since publishing is a rough proxy.

In [ ]:
yt = q("""
SELECT channel_title, title, is_hinglish, subject_group, days_since_publish, view_count,
       SAFE_DIVIDE(view_count, GREATEST(days_since_publish, 1)) AS lifetime_views_per_day,
       views_per_day
FROM `learning-reach-473210.marts.mart_youtube_performance`
WHERE snapshot_date = (SELECT MAX(snapshot_date) FROM `learning-reach-473210.marts.mart_youtube_performance`)
""")
display(yt.groupby(["channel_title", "is_hinglish"])["lifetime_views_per_day"].median().to_frame())
display(yt.groupby("subject_group")["lifetime_views_per_day"].agg(["count", "median"]).sort_values("median", ascending=False))

## Diagnostics summary (fill in)

- Biggest gaps (by gap points): …  Intense small gaps (by ratio): …
- Plan alignment (synthetic plan): …% of planned effort goes to already well-supplied cells.
- Usage-data coverage differs by …  → caveat for statistics.
- Teacher content: …% of all plays.
- Label quality: … has the highest share of titles in another Indian script.